# Week 2. Counting Is Already a Model

**What you'll do today.** You'll build a word-counter by hand-logic, watch the same text get
counted three different ways, scale your count up with tf-idf, run the same counter across three
very different corpora, upgrade the signature-vocabulary trick into proper **keyness** (the
method behind Week 1's featured study), and finish with the one statistics move this course
needs: the **shuffle test** for whether a counted difference is real. (You already counted an
*image*, pixel by pixel, in Week 1; this week goes deep on words.)

> **Don't lose your work.** Opened from GitHub, this notebook is read-only: **File → Save a copy in Drive** before editing, and save durable outputs to your Drive project folder; Colab's own disk is wiped when the runtime ends. Course notebooks get updates during the term; to pick them up, open the notebook fresh from GitHub (or `git pull` if you cloned the repo). Updates never touch your saved copy.


In [ ]:
# If an import fails: re-run the install cell above; if it persists see ../kits/common-errors-cheatsheet.md
# (standalone copy: https://github.com/lucianli123/culture-as-data-2026/blob/main/kits/common-errors-cheatsheet.md)
# --- Make your work survive a Colab reset -------------------------------------
# Colab wipes the runtime when it disconnects or idles out. Mount your Google Drive
# and keep everything in ONE project folder, so your corpus, embeddings, labels, and
# charts are still there next week. (Outside Colab - e.g. the offline test harness -
# this falls back to a local folder so the notebook still runs.)
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/culture-as-data"
except Exception:
    PROJECT_DIR = os.path.abspath("./culture-as-data-project")
os.makedirs(PROJECT_DIR, exist_ok=True)
print("Project folder:", PROJECT_DIR)

In [ ]:
# Fetch the pinned package lists if they aren't beside the notebook (this happens
# whenever you open a single notebook straight from GitHub in Colab).
import os, urllib.request
_RAW = "https://raw.githubusercontent.com/lucianli123/culture-as-data-2026/main/notebooks/"
for _f in ("requirements.txt", "constraints.txt"):
    if not os.path.exists(_f):
        urllib.request.urlretrieve(_RAW + _f, _f)

# First, install the few packages Colab doesn't already ship. If you opened this
# notebook with the whole repo, the line below uses our pinned versions:
%pip install -q -r requirements.txt -c constraints.txt

# Opened this notebook on its own, without the repo files? Comment the line above
# and use this explicit pinned install instead:
# %pip install -q pandas scikit-learn matplotlib pillow

In [ ]:
# Imports. If this cell fails, it almost always means the install cell above didn't run
# this session, re-run it, then re-run this. (See the cheat sheet, entry 5.)
import re
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
print("imports ok")

In [ ]:
import os

## Three real corpora

A *corpus* is just "a pile of text you're studying." Today's three are real and
snapshotted in the repo: Shakespeare's 154 sonnets, and two novels, *Frankenstein*
(1818) and *Dracula* (1897), split into paragraphs. Same counter, three very different
voices.

In [ ]:
import re

def load_gutenberg(path, url):
    """Repo snapshot first, then the live source; strips the Gutenberg boilerplate."""
    if os.path.exists(path):
        raw = open(path, encoding="utf-8-sig").read()
    else:
        import requests
        raw = requests.get(url, timeout=30).text
    body = re.split(r"\*\*\* ?START OF (?:THE|THIS) PROJECT GUTENBERG.*?\*\*\*", raw, flags=re.S)[-1]
    return re.split(r"\*\*\* ?END OF (?:THE|THIS) PROJECT GUTENBERG", body)[0]

def paragraphs(text, min_chars=200):
    return [p.strip().replace("\n", " ") for p in text.split("\n\n") if len(p.strip()) >= min_chars]

_s = load_gutenberg("data/week01/shakespeare_sonnets.txt", "https://www.gutenberg.org/cache/epub/1041/pg1041.txt")
_parts = re.split(r"\n\s*([IVXLC]+)\s*\n", _s)
poems = [_parts[i+1].strip() for i in range(1, len(_parts) - 1, 2) if 200 < len(_parts[i+1].strip()) < 900]
frank = paragraphs(load_gutenberg("data/texts/frankenstein.txt", "https://www.gutenberg.org/cache/epub/84/pg84.txt"))
drac = paragraphs(load_gutenberg("data/texts/dracula.txt", "https://www.gutenberg.org/cache/epub/345/pg345.txt"))
corpora = {"sonnets": poems, "frankenstein": frank, "dracula": drac}
print(len(poems), "sonnets |", len(frank), "Frankenstein paragraphs |", len(drac), "Dracula paragraphs")

## Bag-of-words by hand: counting *requires defining*

Before any library, count by hand-logic.

In [ ]:
def tokenize(text, fold_case=True, strip_punct=True):
    if fold_case:
        text = text.lower()
    if strip_punct:
        text = re.sub(r"[^a-z0-9\s]", " ", text)
    return [t for t in text.split() if t]

sample_text = " ".join(poems[:20])
counts = Counter(tokenize(sample_text))
print("Top words in the first twenty sonnets:")
for word, n in counts.most_common(8):
    print(f"  {word:8s} {n}")

In [ ]:
# A "stop word" list is itself a choice. Watch the top-words list change when we drop
# common function words, the same kind of decision as merging run/running.
STOP = {"i","we","you","it","the","a","and","to","of","in","so","be","on","my","me","now"}
counts_nostop = Counter(t for t in tokenize(sample_text) if t not in STOP)
print("Top words with stop-words removed:")
for word, n in counts_nostop.most_common(8):
    print(f"  {word:8s} {n}")

### What counts as a word? Tokens, not words

Models never see words.

## tf-idf: "common here, rare overall"

Raw counts are dominated by words that are common *everywhere* (*the*, *you*).

In [ ]:
docs = poems + frank[:80] + drac[:80]
labels = ["sonnet"] * len(poems) + ["frankenstein"] * 80 + ["dracula"] * 80
vec = TfidfVectorizer(stop_words="english")
X = vec.fit_transform(docs)
terms = np.array(vec.get_feature_names_out())

# The most distinctive terms of three famous sonnets, by tf-idf:
for i in [0, 17, 129]:
    row = X[i].toarray().ravel()
    print(f"sonnet {i+1:3d}:", ", ".join(terms[row.argsort()[::-1][:4]]))
# "Common here, rare overall": tf-idf finds what marks a document out from the pile.

## Go further: cross-corpus counting (not scheduled in the session)

The corpus choice changes what "common" means. It also works as a corpus sampler before Week 4.

Same counter, three corpora.

In [ ]:
for name, dd in corpora.items():
    text = " ".join(dd)
    top = Counter(t for t in tokenize(text) if t not in STOP).most_common(5)
    print(f"{name:8s}:", ", ".join(f"{w}({n})" for w, n in top))

### A signature vocabulary

Counting alone can show you a *voice*: the words one source uses far more than a baseline.

In [ ]:
# Words the sonnets use much more than the novels, by simple rate ratio.
def rates(text):
    toks = tokenize(text)
    n = len(toks) or 1
    return {w: c / n for w, c in Counter(toks).items()}

sig = rates(" ".join(poems))
base = rates(" ".join(frank[:200]) + " " + " ".join(drac[:200]))
distinctive = sorted(((w, sig[w] / (base.get(w, 0) + 1e-6)) for w, r in sig.items() if sig[w] > 3e-4),
                     key=lambda t: -t[1])[:10]
print("distinctively sonnet-speak:", ", ".join(w for w, _ in distinctive))

## Keyness: distinctively hers, done properly

The ratio above was a rough cut. The standard tool is a **log-odds ratio**: for every word,
how much more likely is it in corpus A than corpus B (with a little smoothing so rare words
don't explode)? Positive means "distinctively A," negative "distinctively B."

**The reveal:** this is *exactly* the method behind Week 1's featured study. *She Giggles, He
Gallops* is a log-odds ratio of stage-direction verbs after "she" vs. "he," run on 2,000
screenplays. You now own the arithmetic behind the piece you admired.

In [ ]:
import numpy as np
from collections import Counter

def counts(texts):
    c = Counter()
    for t in texts:
        c.update(tokenize(t))
    return c

A, B = counts(poems), counts(frank[:200] + drac[:200])   # sonnets vs. the novels
vocab = [w for w in (A | B) if (A[w] + B[w]) >= 3]        # ignore the rarest words
NA, NB = sum(A.values()), sum(B.values())

def log_odds(w):
    pa = (A[w] + 1) / (NA + len(vocab))                   # +1 smoothing
    pb = (B[w] + 1) / (NB + len(vocab))
    return np.log(pa / (1 - pa)) - np.log(pb / (1 - pb))

scored = sorted(vocab, key=log_odds)
print("distinctively SONNETS:", [w for w in scored[-8:][::-1]])
print("distinctively NOVELS: ", [w for w in scored[:8]])

## Is the difference real? The shuffle test

Your top keyness word is more common in the poems than in the baseline. But small corpora
produce flukes. The honest check needs no formulas: **shuffle the labels and count again**.
If documents were randomly dealt into "poems" and "baseline" piles a thousand times, how
often would chance alone produce a gap as big as the real one? Rarely: you have a finding.
Often: you have a coincidence. (And a gap can be real *and* tiny; the shuffle test says
"probably not chance," never "big enough to matter." That's your call to argue.)

In [ ]:
rng = np.random.default_rng(0)
word = scored[-1]                                          # our most "distinctively sonnet" word
docs_all = poems + frank[:80] + drac[:80]
word_counts = np.array([tokenize(d).count(word) for d in docs_all])
totals = np.array([max(1, len(tokenize(d))) for d in docs_all])
labels = np.array([1] * len(poems) + [0] * 160)            # 1 = sonnet, 0 = novel paragraph

def rate_gap(lab):
    return word_counts[lab == 1].sum() / totals[lab == 1].sum() - word_counts[lab == 0].sum() / totals[lab == 0].sum()

observed = rate_gap(labels)
shuffled = []
for _ in range(1000):
    rng.shuffle(labels)
    shuffled.append(rate_gap(labels))

beat = sum(abs(g) >= abs(observed) for g in shuffled)
plt.figure(figsize=(6, 3.5))
plt.hist(shuffled, bins=30)
plt.axvline(observed, color="red", linewidth=2, label=f"the real gap for {word!r}")
plt.title(f"1,000 shuffles: chance matched the real gap {beat} times")
plt.xlabel("gap produced by a random shuffle"); plt.ylabel("shuffles")
plt.legend(); plt.tight_layout(); plt.show()
print(f"chance beat the observed gap in {beat}/1000 shuffles",
      "- probably not chance." if beat < 50 else "- could easily be chance; don't lean on it.")

## Playground: your question

Everything above is a kit. Pick a question and answer it with the pieces: choose any two
corpora (or two slices of one), a word or a pattern, and run keyness plus the shuffle
test on it. Write the question first, in one sentence; the cell is yours to edit.

In [ ]:
# MY QUESTION: ...one sentence here...
GROUP_A = poems               # try: frank, drac, poems[:77], poems[77:]
GROUP_B = drac[:200]          # any other pile of texts

A2, B2 = counts(GROUP_A), counts(GROUP_B)
vocab2 = [w for w in (A2 | B2) if (A2[w] + B2[w]) >= 5]
NA2, NB2 = sum(A2.values()), sum(B2.values())
def lo(w):
    pa = (A2[w] + 1) / (NA2 + len(vocab2)); pb = (B2[w] + 1) / (NB2 + len(vocab2))
    return np.log(pa / (1 - pa)) - np.log(pb / (1 - pb))
sc = sorted(vocab2, key=lo)
print("distinctively A:", sc[-8:][::-1])
print("distinctively B:", sc[:8])
# Now shuffle-test your most distinctive word with the cell above: swap it into `word`,
# rebuild docs_all/labels from YOUR two groups, and see whether chance can match the gap.

## You made a counter, and you can defend it

You defined what a word is, counted three corpora three ways, found the words that make a
voice distinctive with the same method as a famous published study, and shuffle-tested the
gap so you know it isn't chance.

**Sketch (homework):** count something in a text you love; make one chart; write one sentence
naming a choice you made. If your count compares two things, shuffle-test the gap.